In [ ]:
# ============================================================
# Cell 0: Notebook Overview
# ============================================================
#
# Notebook:
# 07_Train_Neural_Network.ipynb
#
# Purpose:
# Train one or more Multi-Layer Perceptron (MLP) classifiers
# using the normalized Digital Image Processing (DIP) feature
# vectors generated in previous pipeline steps.
#
# This notebook focuses exclusively on model training and the
# generation of baseline candidate models for subsequent
# validation and tuning.
#
# ------------------------------------------------------------
# Inputs:
# ------------------------------------------------------------
# Normalized training and validation datasets produced in
# Notebook 06:
#
#   - X_train
#   - y_train
#   - X_validation
#   - y_validation
#
# These datasets:
#   - contain the full DIP feature vectors (~26 features)
#   - are normalized using training-set statistics
#   - are free of missing values
#
# ------------------------------------------------------------
# Assumptions:
# ------------------------------------------------------------
# - Feature extraction and normalization have been completed
# - Training and validation sets are properly separated
# - No information from the test set is used in this notebook
#
# ------------------------------------------------------------
# Cell-by-Cell Flow:
# ------------------------------------------------------------
# Cell 1:
#   Environment setup and imports
#
# Cell 2:
#   Load normalized training and validation data
#
# Cell 3:
#   Run sanity checks on loaded data
#
# Cell 4:
#   Define baseline MLP models using a shared configuration
#   so that all models are identical except for hidden-layer
#   architecture
#
# Cell 5:
#   Train each baseline model, compute basic training and
#   validation accuracy for monitoring, and save trained
#   model files for Notebook 08
#
# Cell 6:
#   Summarize baseline training results in tabular form
#
# Cell 7:
#   Save the baseline training summary for later use in
#   Notebook 08
#
# ------------------------------------------------------------
# Outputs:
# ------------------------------------------------------------
# - Trained MLP baseline model objects
# - Basic training and validation accuracy results
# - Saved model files (e.g., .pkl)
# - Baseline model summary table
# - Saved CSV summary for downstream tuning and comparison
#
# ------------------------------------------------------------
# Notes:
# ------------------------------------------------------------
# - This notebook does NOT perform final model selection
# - Hyperparameter tuning is handled in Notebook 08
# - The test dataset is NOT used at any point in this notebook
# - Validation results here are for sanity checking only and
#   not for final independent reporting
#
# ============================================================


In [ ]:
# ============================================================
# Cell 1: Environment Setup and Imports
# ============================================================

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

# ------------------------------------------------------------
# Standard library imports
# ------------------------------------------------------------
import os
import joblib
import warnings

# ------------------------------------------------------------
# Third-party imports
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------
# Optional display settings
# ------------------------------------------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
BASE_DRIVE_DIR = "/content/drive/MyDrive/DIP_Project"

# Expected inputs from Notebook 06
TRAIN_DATA_PATH = os.path.join(BASE_DRIVE_DIR, "train_feature_vectors_normalized.csv")
VALIDATION_DATA_PATH = os.path.join(BASE_DRIVE_DIR, "validation_feature_vectors_normalized.csv")

# Output directory for trained baseline models
MODEL_OUTPUT_DIR = os.path.join(BASE_DRIVE_DIR, "models")
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

print("Environment setup complete.")
print(f"Base project directory : {BASE_DRIVE_DIR}")
print(f"Training data path     : {TRAIN_DATA_PATH}")
print(f"Validation data path   : {VALIDATION_DATA_PATH}")
print(f"Model output directory : {MODEL_OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment setup complete.
Base project directory : /content/drive/MyDrive/DIP_Project
Training data path     : /content/drive/MyDrive/DIP_Project/train_feature_vectors_normalized.csv
Validation data path   : /content/drive/MyDrive/DIP_Project/validation_feature_vectors_normalized.csv
Model output directory : /content/drive/MyDrive/DIP_Project/models


In [ ]:
# ============================================================
# Cell 2: Load Normalized Training and Validation Data
# ============================================================

print("Loading normalized feature-vector datasets...\n")

# ------------------------------------------------------------
# Load combined normalized datasets
# ------------------------------------------------------------
train_df = pd.read_csv(TRAIN_DATA_PATH)
validation_df = pd.read_csv(VALIDATION_DATA_PATH)

print("Loaded files:")
print(f"Train      : {TRAIN_DATA_PATH}")
print(f"Validation : {VALIDATION_DATA_PATH}")

print("\nShapes:")
print(f"train_df      : {train_df.shape}")
print(f"validation_df : {validation_df.shape}")

# ------------------------------------------------------------
# Define non-feature columns
# ------------------------------------------------------------
NON_FEATURE_COLUMNS = [
    "filename",
    "class_label",
    "source_dataset",
    "subset"
]

# Keep only columns that actually exist
non_feature_cols_present = [col for col in NON_FEATURE_COLUMNS if col in train_df.columns]

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------
X_train_df = train_df.drop(columns=non_feature_cols_present)
X_validation_df = validation_df.drop(columns=non_feature_cols_present)

y_train_raw = train_df["class_label"].copy()
y_validation_raw = validation_df["class_label"].copy()

# ------------------------------------------------------------
# Convert to numpy arrays
# ------------------------------------------------------------
X_train = X_train_df.values
X_validation = X_validation_df.values

# Encode labels: ai / real -> integers
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_validation = label_encoder.transform(y_validation_raw)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------
print("\nFeature columns used:")
print(list(X_train_df.columns))

print("\nData loaded successfully.\n")
print("Final shapes:")
print(f"X_train       : {X_train.shape}")
print(f"X_validation  : {X_validation.shape}")
print(f"y_train       : {y_train.shape}")
print(f"y_validation  : {y_validation.shape}")

print("\nLabel mapping:")
for idx, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {idx}")

print("\nClass distribution (train):")
print(pd.Series(y_train_raw).value_counts())

print("\nClass distribution (validation):")
print(pd.Series(y_validation_raw).value_counts())


Loading normalized feature-vector datasets...

Loaded files:
Train      : /content/drive/MyDrive/DIP_Project/train_feature_vectors_normalized.csv
Validation : /content/drive/MyDrive/DIP_Project/validation_feature_vectors_normalized.csv

Shapes:
train_df      : (8400, 30)
validation_df : (1800, 30)

Feature columns used:
['Mean Gradient', 'Std Gradient', 'Max Gradient', 'Gradient Entropy', 'Edge Density', 'Orientation Mean', 'Orientation Std', 'Orientation Entropy', 'Global Entropy', 'Local Entropy Mean', 'Local Entropy Std', 'Intensity Mean', 'Intensity Std', 'Laplacian Variance', 'Patch Variance Mean', 'Patch Variance Std', 'Noise Residual Energy', 'Low Frequency Energy Ratio', 'Mid Frequency Energy Ratio', 'High Frequency Energy Ratio', 'Radial Mean', 'Radial Std', 'Radial Entropy', 'Spectral Centroid', 'Spectral Bandwidth', 'Log Spectrum Std']

Data loaded successfully.

Final shapes:
X_train       : (8400, 26)
X_validation  : (1800, 26)
y_train       : (8400,)
y_validation  : (1800

In [ ]:
# ============================================================
# Cell 3: Sanity Checks on Loaded Data
# ============================================================

print("Running sanity checks...\n")

# ------------------------------------------------------------
# Basic shape checks
# ------------------------------------------------------------
assert X_train.ndim == 2, "X_train must be a 2D feature matrix"
assert X_validation.ndim == 2, "X_validation must be a 2D feature matrix"
assert y_train.ndim == 1, "y_train must be a 1D label array"
assert y_validation.ndim == 1, "y_validation must be a 1D label array"

assert X_train.shape[0] == y_train.shape[0], "Mismatch between X_train and y_train row counts"
assert X_validation.shape[0] == y_validation.shape[0], "Mismatch between X_validation and y_validation row counts"

assert X_train.shape[1] == X_validation.shape[1], "Train and validation feature counts do not match"

# ------------------------------------------------------------
# Missing-value checks
# ------------------------------------------------------------
assert not np.isnan(X_train).any(), "NaN values found in X_train"
assert not np.isnan(X_validation).any(), "NaN values found in X_validation"
assert not np.isnan(y_train).any(), "NaN values found in y_train"
assert not np.isnan(y_validation).any(), "NaN values found in y_validation"

# ------------------------------------------------------------
# Finite-value checks
# ------------------------------------------------------------
assert np.isfinite(X_train).all(), "Non-finite values found in X_train"
assert np.isfinite(X_validation).all(), "Non-finite values found in X_validation"

# ------------------------------------------------------------
# Class checks
# ------------------------------------------------------------
train_classes = np.unique(y_train)
validation_classes = np.unique(y_validation)

print("Unique classes in y_train      :", train_classes)
print("Unique classes in y_validation :", validation_classes)

assert len(train_classes) == 2, "Expected exactly 2 classes in y_train"
assert len(validation_classes) == 2, "Expected exactly 2 classes in y_validation"

# ------------------------------------------------------------
# Normalization spot check
# ------------------------------------------------------------
train_means = np.mean(X_train, axis=0)
train_stds = np.std(X_train, axis=0)

print("\nNormalization spot check (training set):")
print("Mean (first 5 features):")
print(train_means[:5])

print("\nStd (first 5 features):")
print(train_stds[:5])

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("\nSanity check summary:")
print(f"Training samples       : {X_train.shape[0]}")
print(f"Validation samples     : {X_validation.shape[0]}")
print(f"Number of features     : {X_train.shape[1]}")
print(f"Training label counts  : {pd.Series(y_train).value_counts().to_dict()}")
print(f"Validation label counts: {pd.Series(y_validation).value_counts().to_dict()}")

print("\nAll sanity checks passed.")


Running sanity checks...

Unique classes in y_train      : [0 1]
Unique classes in y_validation : [0 1]

Normalization spot check (training set):
Mean (first 5 features):
[-2.23947844e-16 -2.91618581e-16  3.04941257e-16  3.79379068e-16
 -2.45306421e-17]

Std (first 5 features):
[1. 1. 1. 1. 1.]

Sanity check summary:
Training samples       : 8400
Validation samples     : 1800
Number of features     : 26
Training label counts  : {0: 4200, 1: 4200}
Validation label counts: {0: 900, 1: 900}

All sanity checks passed.


In [ ]:
# ============================================================
# Cell 4: Define Baseline MLP Models (Centralized Config)
# ============================================================

# ------------------------------------------------------------
# Rationale:
# ------------------------------------------------------------
# All baseline models are constructed using a shared configuration
# to ensure that they are identical in every respect except for
# their hidden-layer architecture.
#
# Benefits of this approach:
# - Guarantees a controlled comparison across models
# - Prevents accidental differences in hyperparameters
# - Improves reproducibility and readability
# - Makes future tuning (Notebook 08) easier and less error-prone
#
# Only the model capacity (hidden layer sizes) is varied in this
# notebook to study its effect on performance.
# ------------------------------------------------------------

# ------------------------------------------------------------
# Base configuration (shared across all models)
# ------------------------------------------------------------
MLP_BASE_CONFIG = {
    "activation": "relu",
    "solver": "adam",
    "alpha": 0.0001,
    "batch_size": "auto",
    "learning_rate": "constant",
    "learning_rate_init": 0.001,
    "max_iter": 300,
    "random_state": 42,
    "early_stopping": False
}

# ------------------------------------------------------------
# Model architectures (only varying parameter)
# ------------------------------------------------------------
MODEL_ARCHITECTURES = {
    "MLP_Small": (32,),
    "MLP_Medium": (64, 32),
    "MLP_Large": (128, 64)
}

# ------------------------------------------------------------
# Build baseline models
# ------------------------------------------------------------
BASELINE_MODELS = {}

for model_name, layers in MODEL_ARCHITECTURES.items():
    config = MLP_BASE_CONFIG.copy()
    config["hidden_layer_sizes"] = layers

    BASELINE_MODELS[model_name] = MLPClassifier(**config)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------
print("Baseline models defined (shared configuration):\n")

for model_name, model in BASELINE_MODELS.items():
    print(f"{model_name}:")
    print(f"  hidden_layer_sizes = {model.hidden_layer_sizes}")
    print(f"  activation         = {model.activation}")
    print(f"  solver             = {model.solver}")
    print(f"  alpha              = {model.alpha}")
    print(f"  learning_rate_init = {model.learning_rate_init}")
    print(f"  max_iter           = {model.max_iter}")
    print(f"  random_state       = {model.random_state}")
    print()


Baseline models defined (shared configuration):

MLP_Small:
  hidden_layer_sizes = (32,)
  activation         = relu
  solver             = adam
  alpha              = 0.0001
  learning_rate_init = 0.001
  max_iter           = 300
  random_state       = 42

MLP_Medium:
  hidden_layer_sizes = (64, 32)
  activation         = relu
  solver             = adam
  alpha              = 0.0001
  learning_rate_init = 0.001
  max_iter           = 300
  random_state       = 42

MLP_Large:
  hidden_layer_sizes = (128, 64)
  activation         = relu
  solver             = adam
  alpha              = 0.0001
  learning_rate_init = 0.001
  max_iter           = 300
  random_state       = 42



In [ ]:
# ============================================================
# Cell 5: Train and Save Baseline Models
# ============================================================

# ------------------------------------------------------------
# Purpose:
# ------------------------------------------------------------
# Train each baseline MLP model using the normalized training
# dataset, compute basic validation metrics for monitoring, and
# save the trained model for later comparison in Notebook 08.
#
# Notes:
# - Validation results here are for sanity checking only
# - Final model comparison and selection will occur in Notebook 08
# ------------------------------------------------------------

baseline_results = []

print("Training baseline models...\n")

for model_name, model in BASELINE_MODELS.items():
    print(f"Training {model_name} ...")

    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------
    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------
    y_train_pred = model.predict(X_train)
    y_validation_pred = model.predict(X_validation)

    # --------------------------------------------------------
    # Basic metrics
    # --------------------------------------------------------
    train_accuracy = accuracy_score(y_train, y_train_pred)
    validation_accuracy = accuracy_score(y_validation, y_validation_pred)

    # Loss curves are available for sklearn MLPClassifier after fit
    final_training_loss = model.loss_

    # --------------------------------------------------------
    # Save model
    # --------------------------------------------------------
    model_path = os.path.join(MODEL_OUTPUT_DIR, f"{model_name}.pkl")
    joblib.dump(model, model_path)

    # --------------------------------------------------------
    # Store summary
    # --------------------------------------------------------
    baseline_results.append({
        "model_name": model_name,
        "hidden_layer_sizes": model.hidden_layer_sizes,
        "train_accuracy": train_accuracy,
        "validation_accuracy": validation_accuracy,
        "final_training_loss": final_training_loss,
        "n_iterations": model.n_iter_,
        "model_path": model_path
    })

    # --------------------------------------------------------
    # Status print
    # --------------------------------------------------------
    print(f"  hidden_layer_sizes  : {model.hidden_layer_sizes}")
    print(f"  train_accuracy      : {train_accuracy:.4f}")
    print(f"  validation_accuracy : {validation_accuracy:.4f}")
    print(f"  final_training_loss : {final_training_loss:.6f}")
    print(f"  n_iterations        : {model.n_iter_}")
    print(f"  saved_to            : {model_path}\n")

print("Baseline training complete.")


Training baseline models...

Training MLP_Small ...
  hidden_layer_sizes  : (32,)
  train_accuracy      : 0.9017
  validation_accuracy : 0.8706
  final_training_loss : 0.233440
  n_iterations        : 300
  saved_to            : /content/drive/MyDrive/DIP_Project/models/MLP_Small.pkl

Training MLP_Medium ...
  hidden_layer_sizes  : (64, 32)
  train_accuracy      : 0.9802
  validation_accuracy : 0.8722
  final_training_loss : 0.072627
  n_iterations        : 300
  saved_to            : /content/drive/MyDrive/DIP_Project/models/MLP_Medium.pkl

Training MLP_Large ...
  hidden_layer_sizes  : (128, 64)
  train_accuracy      : 0.9969
  validation_accuracy : 0.8806
  final_training_loss : 0.022265
  n_iterations        : 231
  saved_to            : /content/drive/MyDrive/DIP_Project/models/MLP_Large.pkl

Baseline training complete.


In [ ]:
# ============================================================
# Cell 6: Summarize Baseline Training Results
# ============================================================

# Convert collected results to a DataFrame for easier inspection
baseline_results_df = pd.DataFrame(baseline_results)

# Sort by validation accuracy (highest first)
baseline_results_df = baseline_results_df.sort_values(
    by="validation_accuracy",
    ascending=False
).reset_index(drop=True)

print("Baseline model summary:\n")
display(baseline_results_df)


Baseline model summary:



,model_name,hidden_layer_sizes,train_accuracy,validation_accuracy,final_training_loss,n_iterations,model_path
0,MLP_Large,"(128, 64)",0.996905,0.880556,0.022265,231,/content/drive/MyDrive/DIP_Project/models/MLP_...
1,MLP_Medium,"(64, 32)",0.980238,0.872222,0.072627,300,/content/drive/MyDrive/DIP_Project/models/MLP_...
2,MLP_Small,"(32,)",0.901667,0.870556,0.233440,300,/content/drive/MyDrive/DIP_Project/models/MLP_...


In [ ]:
# ============================================================
# Cell 7: Save Baseline Training Summary
# ============================================================

# ------------------------------------------------------------
# Purpose:
# ------------------------------------------------------------
# Save the baseline training summary table so it can be reused
# later in Notebook 08 for comparison and tuning analysis.
# ------------------------------------------------------------

BASELINE_RESULTS_CSV_PATH = os.path.join(BASE_DRIVE_DIR, "baseline_model_results.csv")

baseline_results_df.to_csv(BASELINE_RESULTS_CSV_PATH, index=False)

print("Baseline training summary saved.")
print(f"Saved to: {BASELINE_RESULTS_CSV_PATH}")


Baseline training summary saved.
Saved to: /content/drive/MyDrive/DIP_Project/baseline_model_results.csv
